# guardbench sur GPU (Google Colab)

Ce notebook relance le benchmark de guardrails sur un GPU Colab (T4 gratuit), avec des échantillons plus grands et des modèles plus lourds que l'exécution CPU de référence : Llama Guard 3 1B et 8B servis par `transformers`, Prompt-Guard-86M.

Les résultats sont écrits dans `results/colab-gpu`, séparément de `results/cpu-baseline`. Les latences GPU et CPU ne se comparent pas entre elles.

Avant de lancer :

1. `Exécution > Modifier le type d'exécution > GPU T4`.
2. Pour les modèles Meta (Llama Guard 3, Prompt-Guard), accepter leur licence sur le Hub avec votre compte, puis ajouter un secret Colab nommé `HF_TOKEN` (icône clé à gauche). Sans ce secret, ces outils sont sautés et la raison est écrite dans `tools_status.json`.
3. Facultatif : un secret `LAKERA_API_KEY` active l'adaptateur Lakera Guard.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone --depth 1 https://github.com/Adam-Blf/llm-guardrails-benchmark.git
%cd llm-guardrails-benchmark
# torch CUDA de Colab est conservé ; le reste suit les extras du projet.
!pip install -q -e ".[tools]" bitsandbytes
!pip install -q --no-deps rebuff==0.1.1
!pip install -q https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl https://github.com/explosion/spacy-models/releases/download/fr_core_news_sm-3.8.0/fr_core_news_sm-3.8.0-py3-none-any.whl

In [ ]:
import os
try:
    from google.colab import userdata
    for key in ("HF_TOKEN", "LAKERA_API_KEY"):
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            print(f"secret {key} absent, outils correspondants sautés")
except ImportError:
    print("hors Colab : définir HF_TOKEN dans l'environnement")

In [ ]:
!python -m guardbench list

## Exécution

250 exemples par classe (au lieu de 100 sur CPU). La tranche d'injection indirecte reste plafonnée par les 50 courriels de test de BIPIA. NeMo Guardrails et la variante Ollama de Llama Guard ne tournent pas ici, faute de serveur Ollama.

In [ ]:
TOOLS = ("regex_baseline presidio protectai_deberta_v2 prompt_guard_86m gliner2_gliguard "
         "gliner2_pii llm_guard llama_guard3_1b_hf llama_guard3_8b_hf rebuff_heuristic lakera_guard")
!python -m guardbench --run colab-gpu run --n-per-class 250 --workers 2 --tools {TOOLS}

In [ ]:
!python -m guardbench --run colab-gpu analyse
!python -m guardbench --run colab-gpu figures --out reports/figures/colab-gpu

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("guardbench-colab-gpu", "zip", ".", "results/colab-gpu")
shutil.make_archive("guardbench-colab-gpu-figures", "zip", ".", "reports/figures/colab-gpu")
files.download("guardbench-colab-gpu.zip")
files.download("guardbench-colab-gpu-figures.zip")

Pour intégrer ces résultats au dépôt, décompresser l'archive à la racine du clone local : `results/colab-gpu/` et `reports/figures/colab-gpu/` s'ajoutent sans toucher à la référence CPU.